# 10 · Geodesic domes on three base polyhedra

`GeodesicDome(f, base=...)` builds the dome on a **tetrahedron**, an **icosahedron** (the default) or a
**dodecahedron**. Every face of the base solid is cut into an *f*-frequency triangular grid and pushed
out onto the unit sphere. All three use the same index grid: the six neighbours of `(x, y)` are always
`(±1, 0)`, `(0, ±1)`, `(+1, +1)` and `(−1, −1)`.

| base | unique points | triangles | corners |
|---|---:|---:|---|
| `'tetrahedron'` (4HSOM array, de Sousa & Oliveira 2012) | 2f²+2 | 4f² | 4, with 3 neighbours |
| `'icosahedron'` (Wu & Takatsuka 2006) | 10f²+2 | 20f² | 12, with 5 neighbours |
| `'dodecahedron'` (pentakis dodecahedron) | 30f²+2 | 60f² | 12 pentagon centres, with 5 neighbours |

Script version: [`examples/10_base_polyhedra.py`](../10_base_polyhedra.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import numpy as np
from dome_utils import shade, unique_mesh
from IPython.display import display
from matplotlib.collections import PolyCollection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from nb_setup import LiveFigure

from mt.geodesicdome.grid.geodesicdome import GeodesicDome
from mt.geodesicdome.grid.polyhedra import BASES

print(BASES)

## Choosing the base

`base` accepts the full names, `'tetra'`, `'icosa'`, `'dodeca'` (any case), or the number of faces of the
solid. The result is an `IcosahedronDome`, `TetrahedronDome` or `DodecahedronDome`; all are `GeodesicDome`s.

In [ ]:
for name in ('tetra', 'icosa', 'dodeca'):
    dome = GeodesicDome(4, base=name)
    print(f'{name:7s} -> {dome!r:48s} {type(dome).__name__:17s} a GeodesicDome: {isinstance(dome, GeodesicDome)}')

## Sizes

Unique points (*N*), triangles and stored vertices (*N* plus the seam copies) for a few frequencies.
For a SOM, pick the frequency that gives about the number of neurons you want: the tetrahedron grows
slowest (2f²), the dodecahedron fastest (30f²).

In [ ]:
print(f"{'f':>3} " + ''.join(f'{b:>30s}' for b in BASES))
print(f"{'':>3} " + ''.join(f"{'points / triangles / stored':>30s}" for _ in BASES))
for f in (1, 2, 3, 4, 6, 8):
    row = []
    for b in BASES:
        dome = GeodesicDome(f, base=b)
        points, faces, _ = unique_mesh(dome)
        row.append(f'{len(points):>10d} / {len(faces):>5d} / {len(dome.get_all_vertices()):>5d}')
    print(f'{f:>3} ' + ''.join(f'{r:>30s}' for r in row))

## The tetrahedron: the 4HSOM orthogonal array

De Sousa & Oliveira store the tessellated tetrahedron in an array of `f + 1` rows by `2f + 1` columns in
which every cell is used. Column `0` and column `2f` are the same points, and the top and bottom rows
fold about their middle points. Below, `A`–`D` are the tetrahedron's corners (`D` is the north pole),
`*` marks a point stored twice and `·` an interior point.

In [ ]:
f = 3
dome = GeodesicDome(f, base='tetrahedron')
corners = {(0, 0): 'A', (2 * f, 0): 'A', (f, 0): 'B', (0, f): 'C', (2 * f, f): 'C', (f, f): 'D'}
for y in range(dome.y_max, -1, -1):
    cells = []
    for x in range(dome.x_max + 1):
        v = dome.get_vertex_at(x, y)
        cells.append(corners.get((x, y), '*' if v.same_vertices else '·'))
    print(f'row {y}:  ' + '  '.join(cells))
print(f'{len(dome.get_all_vertices())} cells, {len(unique_mesh(dome)[0])} distinct points (2f²+2 = {2 * f * f + 2})')

## Domes and nets

Top: the domes on the sphere (drag to turn them when figures are live). Bottom: the nets they are
indexed on, sheared so that every triangle is equilateral. Orange points are seam copies, red ones the
corners of the base solid. The dodecahedron's net is the icosahedron's strip net with its cuts moved to
pentagon edges, so every one of the 60 triangles stays whole.

In [ ]:
SHEAR = np.array([[1.0, 0.0], [-0.5, np.sqrt(3) / 2]])          # grid (x, y) -> plane
COLOUR = {'tetrahedron': (0.80, 0.42, 0.20), 'icosahedron': (0.27, 0.51, 0.71), 'dodecahedron': (0.30, 0.62, 0.38)}


def draw_domes(fig, freq, bases=BASES):
    for n, base in enumerate(bases):
        dome = GeodesicDome(freq, base=base)
        vertices = dome.get_all_vertices()
        stored = dome.get_all_triangles().reshape(-1, 3)        # also assigns vertex.id
        points, faces, index_map = unique_mesh(dome)

        tri = points[faces]
        normals = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
        normals /= np.linalg.norm(normals, axis=1, keepdims=True)
        ax = fig.add_subplot(2, len(bases), n + 1, projection='3d')
        ax.add_collection3d(Poly3DCollection(tri, facecolors=shade(normals)[:, None] * np.array(COLOUR[base]),
                                             edgecolors='white', linewidths=0.3))
        ax.set_xlim(-1, 1), ax.set_ylim(-1, 1), ax.set_zlim(-1, 1)
        ax.set_box_aspect((1, 1, 1))
        ax.view_init(elev=20, azim=35)
        ax.set_axis_off()
        ax.set_title(f"base='{base}'\n{len(points)} points, {len(faces)} triangles", fontsize=10)

        degree = np.zeros(len(points), dtype=int)
        for a, b in {tuple(sorted(e)) for t in faces for e in ((t[0], t[1]), (t[1], t[2]), (t[2], t[0]))}:
            degree[a] += 1
            degree[b] += 1
        corner = degree[index_map] < 6
        seam = np.array([bool(v.same_vertices) for v in vertices]) & ~corner
        net = np.array([[v.x, v.y] for v in vertices], dtype=float) @ SHEAR
        ax = fig.add_subplot(2, len(bases), n + 1 + len(bases))
        ax.add_collection(PolyCollection(net[stored], facecolors=(*COLOUR[base], 0.18),
                                         edgecolors=COLOUR[base], linewidths=0.5))
        ax.scatter(*net[~seam & ~corner].T, s=5, c='#4a4a4a', zorder=3)
        ax.scatter(*net[seam].T, s=12, c='#e6892e', zorder=3)
        ax.scatter(*net[corner].T, s=34, c='#c0392b', marker='p', zorder=4)
        ax.set_aspect('equal')
        ax.autoscale()
        ax.set_axis_off()
        ax.set_title(f'grid {dome.x_max + 1} x {dome.y_max + 1}, {len(vertices)} stored', fontsize=10)
    fig.tight_layout()


live = LiveFigure(figsize=(13, 8))
draw_domes(live.clear(), 4)
live.refresh()
live.show()

## Neighbours at the corners

Every point has 6 neighbours except the corners of the base solid: 3 on the tetrahedron, 5 on the
icosahedron and at the dodecahedron's pentagon centres. The 20 dodecahedron corners themselves have 6.

In [ ]:
for base in BASES:
    dome = GeodesicDome(4, base=base)
    points, faces, index_map = unique_mesh(dome)
    degree = np.zeros(len(points), dtype=int)
    for a, b in {tuple(sorted(e)) for t in faces for e in ((t[0], t[1]), (t[1], t[2]), (t[2], t[0]))}:
        degree[a] += 1
        degree[b] += 1
    corner = next(v for v in dome.get_all_vertices() if degree[index_map[v.id]] < 6)
    dome.unmark_vertices()
    rings = [len(r) for r in dome.get_neighbours_in_distance(corner, 3)]
    values, counts = np.unique(degree, return_counts=True)
    histogram = {int(d): int(c) for d, c in zip(values, counts, strict=True)}
    print(f'{base:13s} points by degree {histogram}   rings around a corner: {rings}')

## Try it

Choose the frequency (and which solids to show). With `ipympl` you can turn each dome with the mouse.

In [ ]:
freq = widgets.IntSlider(value=3, min=1, max=10, description='frequency')
which = widgets.SelectMultiple(options=BASES, value=BASES, description='bases', rows=3)
try_it = LiveFigure(figsize=(13, 8))


def redraw(freq, which):
    fig = try_it.clear()
    if which:
        draw_domes(fig, freq, bases=[b for b in BASES if b in which])
    try_it.refresh()


out = widgets.interactive_output(redraw, dict(freq=freq, which=which))
display(widgets.HBox([freq, which]), out)
try_it.show()